In [69]:
#imports
import numpy as np
import pandas as pd
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

In [70]:
#Logistic regression from scratch

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

class LogisticRegression():
    def __init__(self, lr=0.1, n_iters=1000, penalty=0.001):
        self.lr = lr
        self.n_iters = n_iters
        self.penalty = penalty
        self.weights = None
        self.bias = None

    def fit(self, X, y):
        X = X.toarray()
        n_samples, n_features = X.shape
        self.weights = np.zeros(n_features)
        self.bias = 0

        n_spam = np.sum(y==1)
        n_ham = np.sum(y==0)
        w_spam = n_samples / (2 * n_spam)
        w_ham  = n_samples / (2 * n_ham)
        sample_weights = np.where(y == 1, w_spam, w_ham)

        for _ in range(self.n_iters):
            linear_pred = np.dot(X, self.weights) + self.bias
            predictions = sigmoid(linear_pred)

            dw = (1/n_samples) * np.dot(X.T, sample_weights * (predictions - y)) + self.penalty * self.weights
            db = (1/n_samples) * np.sum(sample_weights * (predictions - y))

            self.weights = self.weights - self.lr * dw
            self.bias    = self.bias    - self.lr * db

    def predict(self, X):
        X = X.toarray()
        linear_pred = np.dot(X, self.weights) + self.bias
        y_pred = sigmoid(linear_pred)
        return [0 if y <= 0.5 else 1 for y in y_pred]

In [71]:
#Data and processing
def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"[^a-z\s]", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

df = pd.read_csv("/content/SMSSpamCollection",
                 sep="\t",
                 header=None,
                 names=["label", "text"])

df["clean_text"] = df["text"].apply(clean_text)
df["label_num"]  = df["label"].map({"ham": 0, "spam": 1})

print(f"Total  : {len(df)}")
print(f"Ham    : {(df.label == 'ham').sum()}")
print(f"Spam   : {(df.label == 'spam').sum()}")

Total  : 5572
Ham    : 4825
Spam   : 747


In [72]:
#TF-IIDF and Train/Test split
X_train, X_test, y_train, y_test = train_test_split(
    df["clean_text"], df["label_num"],
    test_size=0.2, random_state=0,
    stratify=df["label_num"]
)

vectorizer = TfidfVectorizer(max_features=3000,
                             ngram_range=(1, 2),
                             sublinear_tf=True)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf  = vectorizer.transform(X_test)

print(f"Train : {X_train_tfidf.shape}")
print(f"Test  : {X_test_tfidf.shape}")

Train : (4457, 3000)
Test  : (1115, 3000)


In [73]:
model = LogisticRegression(lr=0.1, n_iters=3000, penalty=0.001)
model.fit(X_train_tfidf, np.array(y_train))

y_pred = model.predict(X_test_tfidf)

print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(classification_report(y_test, y_pred, target_names=["Ham", "Spam"]))

Accuracy: 0.9659
              precision    recall  f1-score   support

         Ham       0.98      0.98      0.98       966
        Spam       0.86      0.89      0.87       149

    accuracy                           0.97      1115
   macro avg       0.92      0.93      0.93      1115
weighted avg       0.97      0.97      0.97      1115

